# 03 — Cluster dynamics
An exploratory decomposition of the SciPy reference step, with the same draw order. FK bonds are not geometric spin clusters. For agreeing edges $p=1-e^{-2\beta}$; plus ghost bonds use $p_g=1-e^{-2\beta h}$. The size rule instead recolors free components of area $A$ with probability $(1+e^{-2\beta h A})^{-1}$. Run the entire sequence after changing `use_ghost` or `seed`; the intermediate cells consume RNG draws or append edges.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
from ising import IsingModel, Simulator
import ising.kernels as kernels
import ising.observables as obs
from ising.state import make_state
import inspect
from scipy.sparse import coo_matrix
from scipy.sparse.csgraph import connected_components

In [ ]:
spin_cmap = ListedColormap(["#14263a", "#2c4a6b", "#e3952f"])
def show_state(spins, ax=None, title=""):
    if ax is None:
        _, ax = plt.subplots(figsize=(4, 4))
    image = spins.copy().astype(int)
    image[0, :] = image[-1, :] = -2
    image[:, 0] = image[:, -1] = -2
    ax.imshow(image, cmap=spin_cmap, vmin=-2, vmax=1, interpolation="nearest")
    ax.set(title=title, xticks=[], yticks=[])
    return ax

In [ ]:
N, beta, h, seed = 6, .55, .2, 42
use_ghost = True  # False selects size-based recoloring
spins = make_state(N, 'minus', np.random.default_rng(seed))
spins[1:4,1:4] = 1; spins[4:6,4:6] = 1
rng = np.random.default_rng(seed)
num_sites = N*N
MINUS, PLUS = num_sites, num_sites+1
bond_prob = -np.expm1(-2*beta); ghost_bond_prob = -np.expm1(-2*beta*h)
interior = spins[1:-1,1:-1]
site_ids = np.arange(num_sites).reshape(N,N)
show_state(spins); plt.show()
print(site_ids)

## Interior bonds
Slicing aligns adjacent sites; Boolean indexing extracts endpoint arrays. The reference draws a uniform even for ineligible edges.

In [ ]:
horizontal_agreement = interior[:,:-1] == interior[:,1:]
vertical_agreement = interior[:-1,:] == interior[1:,:]
horizontal_bonds = horizontal_agreement & (rng.random((N,N-1)) < bond_prob)
vertical_bonds = vertical_agreement & (rng.random((N-1,N)) < bond_prob)
print('eligible horizontal:\n',horizontal_agreement.astype(int),'\nactive horizontal:\n',horizontal_bonds.astype(int))
edge_u = [site_ids[:,:-1][horizontal_bonds], site_ids[:-1,:][vertical_bonds]]
edge_v = [site_ids[:,1:][horizontal_bonds], site_ids[1:,:][vertical_bonds]]
print(edge_u,edge_v)

In [ ]:
ax = show_state(spins,title='Activated interior bonds')
for a,b in zip(np.concatenate(edge_u),np.concatenate(edge_v)):
    i,j = divmod(int(a),N); k,l = divmod(int(b),N)
    ax.plot([j+1,l+1],[i+1,k+1],color='white',lw=2)
plt.show()

## Boundary and optional ghost edges
Each side is tested independently, including repeated corner vertices. Auxiliary vertices represent fixed spins, not interior area. The following cell resets the edge lists to their interior entries but consumes new uniforms; rerun from setup for a reproducible comparison.

In [ ]:
edge_u = edge_u[:2]; edge_v = edge_v[:2]
minus_mask = interior == -1
for boundary_minus_mask, boundary_site_ids in ((minus_mask[:,0],site_ids[:,0]),(minus_mask[:,-1],site_ids[:,-1]),(minus_mask[0,:],site_ids[0,:]),(minus_mask[-1,:],site_ids[-1,:])):
    active_boundary_bonds = boundary_minus_mask & (rng.random(N)<bond_prob)
    edge_u.append(boundary_site_ids[active_boundary_bonds]); edge_v.append(np.full(int(active_boundary_bonds.sum()),MINUS))
if use_ghost:
    active_ghost_bonds = (interior==1) & (rng.random((N,N))<ghost_bond_prob)
    edge_u.append(site_ids[active_ghost_bonds]); edge_v.append(np.full(int(active_ghost_bonds.sum()),PLUS))
u,v = np.concatenate(edge_u),np.concatenate(edge_v)
print(np.column_stack((u,v)))

## Sparse graph and component sizes
The adjacency stores one oriented entry per sampled edge; `directed=False` interprets it as undirected. Duplicate corner edges are harmless for connectivity. Dense conversion is only for this small demonstration.

In [ ]:
num_vertices = num_sites+1+int(use_ghost)
graph = coo_matrix((np.ones(len(u),dtype=np.int8),(u,v)),shape=(num_vertices,num_vertices))
num_components, component_labels = connected_components(graph,directed=False)
component_sizes = np.bincount(component_labels[:num_sites],minlength=num_components)
print('sizes:',component_sizes,'fixed minus label:',component_labels[MINUS])
if use_ghost: print('fixed plus label:',component_labels[PLUS]); assert component_labels[PLUS]!=component_labels[MINUS]
fig, axes = plt.subplots(1,2,figsize=(9,4))
axes[0].imshow(graph.toarray()!=0,cmap='Greys'); axes[0].set_title('Stored adjacency')
axes[1].imshow(component_labels[:num_sites].reshape(N,N),cmap='tab20'); axes[1].set_title('Component labels'); plt.show()

## Recolor and verify
Use the same random draw order as the reference, including draws for components whose colors are subsequently forced.

In [ ]:
plus_probabilities = np.full(num_components,.5) if use_ghost else 1/(1+np.exp(-2*beta*h*component_sizes))
plus_probabilities[component_labels[MINUS]]=0
if use_ghost:
    plus_probabilities[component_labels[PLUS]]=1
    component_spins = rng.integers(0,2,num_components).astype(np.int8)*2-1
    component_spins[component_labels[PLUS]]=1
else:
    component_spins = np.where(rng.random(num_components)<1/(1+np.exp(-2*beta*h*component_sizes)),1,-1).astype(np.int8)
component_spins[component_labels[MINUS]]=-1
print('label, area, P(plus), color:\n',np.column_stack((np.arange(num_components),component_sizes,plus_probabilities,component_spins)))
result = spins.copy(); result[1:-1,1:-1]=component_spins[component_labels[:num_sites]].reshape(N,N)
reference=spins.copy()
kernel=kernels.sw_ghost_python if use_ghost else kernels.sw_size_python
kernel(reference,beta,h,np.random.default_rng(seed))
np.testing.assert_array_equal(result,reference)
fig,axes=plt.subplots(1,2,figsize=(7,4))
show_state(spins,axes[0],'Before'); show_state(result,axes[1],'After'); plt.show()

## Union–find on the same graph
Use the actual source helpers in Python mode (`py_func`) to inspect parent pointers. Union by size and path halving avoid building a sparse graph. Component representatives need not equal SciPy labels; compare partitions instead.

In [ ]:
parent=np.arange(num_vertices); union_find_sizes=np.ones(num_vertices,dtype=np.int64)
for a,b in zip(u,v):
    kernels._join.py_func(parent,union_find_sizes,a,b)
roots=np.array([kernels._root.py_func(parent,a) for a in range(num_vertices)])
np.testing.assert_array_equal(roots[:,None]==roots[None,:],component_labels[:,None]==component_labels[None,:])
print('parent:',parent,'\nroots:',roots,'\nroot sizes:',union_find_sizes[np.unique(roots)])
print(inspect.getsource(kernels._root.py_func))
print(inspect.getsource(kernels._join.py_func))

## Production backends
The Numba kernel samples edges in a different order. Same seeds need not produce the same trajectory. Check invariants, not path equality.

In [ ]:
for ghost_mode in (False,True):
    for backend in ('python','numba'):
        kernel=kernels.CLUSTER_KERNELS[('sw-ghost' if ghost_mode else 'sw-size',backend)]
        actual=spins.copy(); kernel(actual,beta,h,np.random.default_rng(seed))
        assert np.isin(actual,[-1,1]).all()
        assert np.all(actual[[0,-1],:]==-1) and np.all(actual[:,[0,-1]]==-1)
print(inspect.getsource(kernels._sw_step_numba.py_func))